In [ ]:
# ===== Colab环境配置 =====
# 运行此cell安装所有依赖（约2-3分钟）
!pip install -q numpy scipy matplotlib
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q soundfile librosa
!pip install -q deep-filter
!pip install -q pesq pystoi
print('环境配置完成！')

# 确认GPU
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU不可用，请在运行时类型中选择T4 GPU')

# 下载DeepFilterNet代码
import os
if not os.path.exists('DeepFilterNet-main'):
    print('正在克隆DeepFilterNet仓库...')
    !git clone https://github.com/Rikorose/DeepFilterNet.git DeepFilterNet-main
    print('克隆完成！')

# 下载预训练模型
model_dir = 'DeepFilterNet-main/models'
if not os.path.exists(os.path.join(model_dir, 'DeepFilterNet3')):
    print('正在下载预训练模型...')
    !cd DeepFilterNet-main/models && unzip -q DeepFilterNet3.zip 2>/dev/null || echo '需要手动上传模型权重'

# 生成测试音频
print('环境准备就绪！')


# 模块5 第3次课（选修）：DeepFilterNet + CI结合实验本notebook包含：1. 实验设计方法论回顾2. 实验1：修改ERB频带数量3. 实验2：DeepFilterNet -> ACE流水线4. 实验3：不同SNR下的增强效果评估5. 实时CI应用的延迟分析> "修改模型不是瞎改——每一个改动都需要一个假设。"---

## §1 实验设计回顾好的修改实验需要：1. **假设**：基于对模型和问题的理解提出2. **修改方案**：改什么、怎么改3. **基线**：未修改的模型作为参照4. **评估指标**：用什么指标衡量效果5. **分析**：结果是否支持假设？

In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport os, sysDFN_DIR = os.path.join('DeepFilterNet-main', 'DeepFilterNet')ACE_DIR = os.path.join('..', '..', 'module4-deepace', 'ACE')sys.path.insert(0, DFN_DIR)print("模块5 第3次课：DeepFilterNet + CI结合实验")print()# 检查可用工具try:    from df.config import config    config.use_defaults()    print("[OK] DeepFilterNet 可用")    DF_AVAILABLE = Trueexcept:    print("[--] DeepFilterNet 不可用")    DF_AVAILABLE = Falsetry:    sys.path.insert(0, ACE_DIR)    from ace_strategy import ace_strategy    print("[OK] ACE策略可用")    ACE_AVAILABLE = Trueexcept:    print("[--] ACE策略不可用")    ACE_AVAILABLE = False

## §2 实验1：ERB频带数量分析### 假设改变ERB频带数量会影响频率分辨率。频带越少 = 频率表示越粗糙，但可能对噪声更鲁棒。这与CI电极数量的折中类似。### 修改方案对比不同频带数的ERB滤波器组：16、32、48。

In [ ]:
# 对比不同ERB频带数def erb_filterbank(n_erbs, sr, n_fft):    n_freqs = n_fft // 2 + 1    freqs = np.linspace(0, sr // 2, n_freqs)    erb_points = 21.4 * np.log10(0.00437 * freqs + 1)    erb_grid = np.linspace(erb_points[0], erb_points[-1], n_erbs + 2)    hz_grid = (10 ** (erb_grid / 21.4) - 1) / 0.00437    fb = np.zeros((n_erbs, n_freqs))    for i in range(n_erbs):        f_left = hz_grid[i]        f_center = hz_grid[i + 1]        f_right = hz_grid[i + 2]        for j in range(n_freqs):            if freqs[j] >= f_left and freqs[j] <= f_center:                fb[i, j] = (freqs[j] - f_left) / (f_center - f_left + 1e-8)            elif freqs[j] > f_center and freqs[j] <= f_right:                fb[i, j] = (f_right - freqs[j]) / (f_right - f_center + 1e-8)    return fbsr = 48000n_fft = 960fig, axes = plt.subplots(1, 3, figsize=(15, 4))for idx, n_erbs in enumerate([16, 32, 48]):    fb = erb_filterbank(n_erbs, sr, n_fft)    axes[idx].imshow(fb, aspect='auto', origin='lower', cmap='hot',                     extent=[0, sr//2, 0, n_erbs])    axes[idx].set_title("ERB: %d个频带" % n_erbs)    axes[idx].set_xlabel("频率 (Hz)")    axes[idx].set_ylabel("频带")plt.tight_layout()plt.show()print("分析:")print("  16个频带：较粗糙，类似于早期CI处理器")print("  32个频带：DeepFilterNet默认值，良好的平衡")print("  48个频带：更精细的分辨率，接近正常听力")print()print("CI关联：现代CI有22个电极（介于16和32个ERB频带之间）")

## §3 实验2：Enhance-then-Encode 流水线### 假设在ACE编码之前施加语音增强，应该能改善通道选择的准确性，因为增强后的语音中噪声能量对频带的污染更少。### 流水线设计```带噪语音 -> DeepFilterNet -> 增强语音 -> ACE策略 -> 电极图    (模块5)                              (模块4)```对比基线：```带噪语音 -> ACE策略 -> 电极图（无增强）```

In [ ]:
# 模拟 Enhance-then-Encode 流水线# 生成测试信号sr_ace = 16000  # ACE使用16kHzduration = 2.0t = np.linspace(0, duration, int(sr_ace * duration), endpoint=False)# 模拟干净语音f0 = 150clean = np.zeros_like(t)for h in range(1, 8):    clean += (0.4 / h) * np.sin(2 * np.pi * f0 * h * t)clean = clean / np.max(np.abs(clean)) * 0.8# 加噪声np.random.seed(42)noise = np.random.randn(len(t)) * 0.5clean_power = np.mean(clean ** 2)noise_power = np.mean(noise ** 2)snr_db = 5noise_scaled = noise * np.sqrt(clean_power / (noise_power * 10 ** (snr_db / 10) + 1e-8))noisy = clean + noise_scalednoisy = noisy / np.max(np.abs(noisy)) * 0.9# 用维纳滤波作为"增强"from scipy.fft import fft, ifftnoisy_fft = fft(noisy)noise_fft = fft(noise_scaled)clean_fft = fft(clean)# Oracle维纳滤波（最好情况）wiener = (np.abs(clean_fft)**2) / (np.abs(clean_fft)**2 + np.abs(noise_fft)**2 + 1e-8)enhanced = np.real(ifft(wiener * noisy_fft))fig, axes = plt.subplots(3, 1, figsize=(12, 6))axes[0].plot(t[:3200], clean[:3200])axes[0].set_title("干净语音")axes[1].plot(t[:3200], noisy[:3200])axes[1].set_title("带噪语音 (SNR=%d dB)" % snr_db)axes[2].plot(t[:3200], enhanced[:3200])axes[2].set_title("增强后 (维纳滤波)")for ax in axes:    ax.set_xlabel("时间 (s)")plt.tight_layout()plt.show()print("增强质量:")si_sdr_noisy = 10 * np.log10(np.mean(clean**2) / np.mean((noisy - clean)**2 + 1e-8))si_sdr_enhanced = 10 * np.log10(np.mean(clean**2) / np.mean((enhanced - clean)**2 + 1e-8))print("  带噪 SI-SDR: %.1f dB" % si_sdr_noisy)print("  增强 SI-SDR: %.1f dB" % si_sdr_enhanced)

## §4 实验3：SNR扫描评估### 假设增强效果在极低SNR时下降。存在一个"SNR阈值"，低于此阈值时增强几乎没有帮助。

In [ ]:
# SNR扫描评估snr_levels = [-10, -5, 0, 5, 10, 15, 20]results = []for snr in snr_levels:    noise_scaled = noise * np.sqrt(clean_power / (noise_power * 10 ** (snr / 10) + 1e-8))    noisy_snr = clean + noise_scaled    noisy_snr = noisy_snr / (np.max(np.abs(noisy_snr)) + 1e-8) * 0.9    # 维纳增强    n_fft_sig = fft(noisy_snr)    c_fft = fft(clean)    n_fft_est = fft(noise_scaled)    wiener_gain = (np.abs(c_fft)**2) / (np.abs(c_fft)**2 + np.abs(n_fft_est)**2 + 1e-8)    enh = np.real(ifft(wiener_gain * n_fft_sig))    # SI-SDR    si_snr_in = 10 * np.log10(np.mean(clean**2) / np.mean((noisy_snr - clean)**2 + 1e-8))    si_snr_out = 10 * np.log10(np.mean(clean**2) / np.mean((enh - clean)**2 + 1e-8))    improvement = si_snr_out - si_snr_in    results.append((snr, si_snr_in, si_snr_out, improvement))    print("SNR=%3ddB: 输入SI-SDR=%6.1f, 输出SI-SDR=%6.1f, 提升=%+.1f dB" % (        snr, si_snr_in, si_snr_out, improvement))# 绘图snrs = [r[0] for r in results]si_in = [r[1] for r in results]si_out = [r[2] for r in results]improvements = [r[3] for r in results]fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))ax1.plot(snrs, si_in, 'o-', label='输入 (带噪)')ax1.plot(snrs, si_out, 's-', label='输出 (增强)')ax1.set_xlabel('SNR (dB)')ax1.set_ylabel('SI-SDR (dB)')ax1.set_title('输入 vs 输出质量')ax1.legend()ax1.grid(True, alpha=0.3)ax2.bar(range(len(snrs)), improvements, tick_label=[str(s) for s in snrs])ax2.set_xlabel('输入SNR (dB)')ax2.set_ylabel('提升 (dB)')ax2.set_title('不同SNR下的增强提升量')ax2.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## §5 延迟分析### CI/助听器的实时性要求| 应用场景 | 最大延迟 | DeepFilterNet算法延迟 ||---------|---------|---------------------|| 助听器 | 5-10 ms | ~5 ms (hop_size/sr = 480/48000) || CI处理器 | 5-10 ms | ~5 ms || 电话 | ~150 ms | ~5 ms || 视频会议 | ~200 ms | ~5 ms |DeepFilterNet的算法延迟 = hop_size / sr = 480 / 48000 = 10 ms但使用50%重叠，有效延迟约为5 ms。**CI部署面临的挑战：**1. **计算预算**：CI处理器的算力极其有限（约为智能手机的1/10）2. **功耗**：必须靠电池运行12小时以上3. **模型大小**：DeepFilterNet3约有150万参数4. **定点运算**：CI硬件可能不支持浮点运算**研究方向**：模型压缩、量化、剪枝，以实现CI部署。

## §6 讨论：从"语音增强"到"CI语音增强"### 差距在哪里通用语音增强（SE）为普通听力者优化。CI用户有独特的需求：| 通用SE | CI专用SE ||-------|----------|| 优化PESQ | 优化STOI（可懂度） || 全频带增强 | 聚焦CI相关频率 || 任意噪声类型 | CI典型环境（餐厅、街道） || 单一输出 | 必须接入CI编码器（ACE） || 低延迟：锦上添花 | 低延迟：必须 <10ms |### 弥合差距的方向潜在研究方向：1. **CI感知损失函数**：在训练损失中包含ACE通道选择质量2. **联合优化**：端到端训练SE + CI编码器3. **逐电极增强**：对每个CI通道施加不同的增强4. **噪声类型自适应**：聚焦CI用户最困扰的噪声类型5. **CI用户主观评估**：金标准——让CI用户直接评价---## 小结本节课我们：1. 分析了ERB频带数量与CI的关联2. 设计了Enhance-then-Encode流水线3. 进行了SNR扫描评估实验4. 分析了实时CI应用的延迟要求5. 讨论了通用SE与CI专用SE之间的差距**课后任务**：- 基础：运行DeepFilterNet推理，处理3段不同SNR的语音，计算指标- 进阶：构建完整的"增强->ACE"流水线，对比有/无增强时通道选择的差异